# 🛡️ HackOps RL Training Pipeline

This notebook manages the full lifecycle of training Reinforcement Learning agents for the DVWA environment using **Stable-Baselines3** and **CybORG**.

## 📋 Workflow
1.  **Baseline Evaluation**: Establish performance metrics with random agents.
2.  **Blue Agent Training**: Train the defender against random attacks.
3.  **Red Agent Training**: Train the attacker against random defenses.
4.  **Comparative Analysis**: Compare trained models against baselines and each other.
5.  **Adversarial Training**: Train agents against each other (Advanced).

In [1]:
# Force reload of helper modules to pick up changes
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Ensure api directory is in path
if 'api' not in os.getcwd():
    os.chdir('api')
    print(f"Changed directory to: {os.getcwd()}")

import notebook_training_helper
from notebook_training_helper import (
    run_baseline_evaluation, 
    run_training_with_live_progress, 
    plot_training_results, 
    get_available_attacks
)
from training_results import TrainingResults
from training_visualizer import TrainingVisualizer

print("✅ Environment Ready")

✅ Environment Ready


## 1. 📉 Establish Baselines
Before training, we evaluate random agents to set a performance baseline. This gives us a reference point to measure our RL agent's improvement.

In [2]:
# Run Blue Baseline
blue_baseline = run_baseline_evaluation(agent='blue', n_episodes=50)


📉 Running Baseline Evaluation: BLUE
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


KeyboardInterrupt: 

In [ ]:
# Run Red Baseline
red_baseline = run_baseline_evaluation(agent='red', n_episodes=50)

## 2. 🛡️ Train Blue Agent (Defender)
Train the Blue agent to patch vulnerabilities. It starts against a Random Red Agent.

**Configuration:**
- `PPO`: Standard stable policy gradient algorithm.
- `ent_coef`: Controls exploration (0.01 = moderate exploration).

In [ ]:
# Training Configuration
BLUE_CONFIG = {
    'agent': 'blue',
    'algorithm': 'PPO',
    'timesteps': 25000,
    'lr': 0.0003,
    'ent_coef': 0.05,
    'attacks': ['xss', 'sqli']
}

print("Training Blue Agent...")
run_training_with_live_progress(**BLUE_CONFIG)

### 📊 Blue Agent Analysis
Below are the training curves and a comparison against the random baseline.

In [ ]:
plot_training_results('blue')

## 3. ⚔️ Train Red Agent (Attacker)
Train the Red agent to exploit vulnerabilities. It starts against a Random Blue Agent.

**Configuration:**
- `PPO`: Good for continuous/discrete action spaces.
- `ent_coef`: Higher (0.02) to encourage trying different attacks.

In [ ]:
# Training Configuration
RED_CONFIG = {
    'agent': 'red',
    'algorithm': 'PPO',
    'timesteps': 25000,
    'lr': 0.0003,
    'ent_coef': 0.05,
    'attacks': ['xss', 'sqli']
}

print("Training Red Agent...")
run_training_with_live_progress(**RED_CONFIG)

### 📊 Red Agent Analysis
Below are the training curves and a comparison against the random baseline.

In [ ]:
plot_training_results('red')

## 4. 📈 Comparative Analysis
Let's compare the performance of our trained agents.

In [ ]:
results = TrainingResults()
viz = TrainingVisualizer()

try:
    # Load latest sessions
    blue_sessions = results.load_all_sessions('blue')
    red_sessions = results.load_all_sessions('red')
    
    if blue_sessions and red_sessions:
        latest_blue = blue_sessions[0]
        latest_red = red_sessions[0]
        
        print(f"Blue Agent Max Reward: {max(latest_blue['episode_rewards']):.2f}")
        print(f"Red Agent Max Reward: {max(latest_red['episode_rewards']):.2f}")
        
        # Plot Comparison
        viz.plot_comparison(
            [latest_blue, latest_red], 
            title="Blue vs Red Learning Progress",
            save_name="blue_vs_red_comparison.png"
        )
        
        # Display
        from IPython.display import Image, display
        display(Image('plots/blue_vs_red_comparison.png'))
    else:
        print("Run training for both agents first to see comparison.")
except Exception as e:
    print(f"Comparison unavailable: {e}")

## 5. 🔁 Adversarial Training (Next Steps)
Now that both agents are competent, we can pit them against each other.

### Phase A: Blue vs Trained Red

In [ ]:
# Train Blue against the Red model we just trained
run_training_with_live_progress(
    agent='blue',
    algorithm='PPO',
    timesteps=10000,
    lr=0.0001, # Lower learning rate for fine-tuning
    opponent_path='models/red_agent_final.zip'
)

### Phase B: Red vs Trained Blue

In [ ]:
# Train Red against the Blue model we just trained
run_training_with_live_progress(
    agent='red',
    algorithm='PPO',
    timesteps=10000,
    lr=0.0001,
    opponent_path='models/blue_agent_final.zip'
)